# Building Energy Consumption Prediction: Batch ML Pipeline


## Overview

This notebook implements the **batch machine learning pipeline** for predicting hourly building energy consumption using Apache PySpark.

### Dataset
| File | Description |
|------|-------------|
| `data/meters.csv` | Hourly energy meter readings per building |
| `data/building_information.csv` | Building attributes (primary use, square footage, floor count, year built) |
| `data/weather.csv` | Hourly weather readings per site (temperature, wind speed, cloud coverage, pressure) |

### Pipeline Summary
1. **Data Loading** — Define schemas and load three CSVs into Spark DataFrames
2. **Aggregation** — Resample raw readings into 6-hour intervals
3. **Imputation** — Fill missing weather values using median imputation
4. **Feature Engineering** — One-hot encoding, cyclical time encoding, log transforms
5. **Model Training** — Compare Random Forest vs Gradient Boosted Trees
6. **Hyperparameter Tuning** — CrossValidator with ParamGridBuilder
7. **Model Persistence** — Save best pipeline to `models/energy_pipeline_model` for use by the streaming notebook


# Part 1: Data Loading, Transformation and Exploration <a class="anchor" name="part-1"></a>
## 1.1 Data Loading
In this section, you must load the given datasets into PySpark DataFrames and use DataFrame functions to process the data. For plotting, various visualisation packages can be used, but please ensure that you have included instructions to install the additional packages and that the installation will be successful in the provided Docker container (in case your marker needs to clear the notebook and rerun it).

### 1.1.1 Data Loading <a class="anchor" name="1.1"></a>
1.1.1 Write the code to create a SparkSession. For creating the SparkSession, you need to use a SparkConf object to configure the Spark app with a proper application name, to ensure the maximum partition size does not exceed 32MB, and to run locally with all CPU cores on your machine

In [ ]:
import os
# import the needed libraries
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.ml.feature import Imputer
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pyspark.ml.feature import OneHotEncoder, StringIndexer
import math
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
# --- Shared pipeline package: single source of truth for features + model ---
# The model-building cells below delegate to energy_pipeline so the batch and
# streaming jobs share ONE feature implementation (no training/serving skew).
import sys
sys.path.insert(0, "src")
os.environ.setdefault("ARTIFACT_ROOT", "models")  # -> models/energy_pipeline_model
from energy_pipeline import datasets, features, metrics
from energy_pipeline.config import load_config
config = load_config()

In [ ]:
master = "local[*]"
app_name = "building-energy-batch"

# setup configuration parameters for Spark
spark_conf = (SparkConf().setMaster(master).setAppName(app_name).set("spark.sql.files.maxPartitionBytes", "32m"))

# create SparkSession
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()

1.1.2 Write code to define the schemas for the datasets, following the data types suggested in the metadata file. 

In [ ]:
# meters.csv
meters_schema = StructType([
    StructField("building_id", IntegerType(), True),
    StructField("meter_type", StringType(), True),
    StructField("ts", TimestampType(), True),
    StructField("value", DoubleType(), True),
    StructField("row_id", IntegerType(), True)
])

In [ ]:
# buildings.csv
buildings_schema = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("building_id", IntegerType(), True),
    StructField("primary_use", StringType(), True),
    StructField("square_feet", IntegerType(), True),
    StructField("floor_count", IntegerType(), True),
    StructField("row_id", IntegerType(), True),
    StructField("year_built", IntegerType(), True),
    StructField("latent_y", DoubleType(), True),
    StructField("latent_s", DoubleType(), True),
    StructField("latent_r", DoubleType(), True)
])

In [ ]:
# weather.csv
weather_schema = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("air_temperature", DoubleType(), True),
    StructField("cloud_coverage", DoubleType(), True),
    StructField("dew_temperature", DoubleType(), True),
    StructField("sea_level_pressure", DoubleType(), True),
    StructField("wind_direction", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True)
])

1.1.3 Using your schemas, load the CSV files into separate data frames. Print the schemas of all data frames. 

In [ ]:
# meters
meters_df = spark.read.csv("data/meters.csv",header=True,schema=meters_schema)
print("Meters DataFrame Schema:")
meters_df.printSchema()

In [ ]:
# buildings
buildings_df = spark.read.csv("data/building_information.csv",header=True,schema=buildings_schema)
print("Buildings DataFrame Schema:")
buildings_df.printSchema()

In [ ]:
# weather
weather_df = spark.read.csv("data/weather.csv",header=True,schema=weather_schema)
print("Weather DataFrame Schema:")
weather_df.printSchema()

### 1.2 Data Transformation to Create Features <a class="anchor" name="1.2"></a>
In this section, we primarily have three tasks:  
1.2.1 The dataset includes sensors with hourly energy measurements. However, as a grid operator, we don’t need this level of granularity and lowering it can reduce the amount of data we need to process. For each building, we will aggregate the metered energy consumption in 6-hour intervals (0:00-5:59, 6:00-11:59, 12:00-17:59, 18:00-23:59). This will be our target (label) column for this prediction. Perform the aggregation for each building.


In [ ]:
meters_df.show()

In [ ]:
# aggregate 6-hour intervals for energy consumption
meters_labeled = (
    meters_df
    .withColumn("year", F.year("ts"))
    .withColumn("month", F.month("ts"))
    .withColumn("day", F.dayofmonth("ts"))
    .withColumn("hour", F.hour("ts"))
    .withColumn("time_interval", 
                F.when((F.col("hour") >= 0) & (F.col("hour") < 6), "0:00-5:59")
                 .when((F.col("hour") >= 6) & (F.col("hour") < 12), "6:00-11:59")
                 .when((F.col("hour") >= 12) & (F.col("hour") < 18), "12:00-17:59")
                 .otherwise("18:00-23:59")
               )
    .groupBy("building_id","meter_type", "year", "month", "day", "time_interval")
    .agg(F.sum("value").alias("energy_consumption"))
)

meters_labeled.show(10)

In the weather dataset, there are some missing values (null or empty strings). It may lower the quality of our model. Imputation is a way to deal with those missing values. Imputation is the process of replacing missing values in a dataset with substituted, or "imputed," values. It's a way to handle gaps in your data so that you can still analyse it effectively without having to delete incomplete records.  
1.2.2 Refer to the Spark MLLib imputation API and fill in the missing values in the weather dataset. You can use mean values as the strategy.  https://spark.apache.org/docs/3.5.5/api/python/reference/api/pyspark.ml.feature.Imputer.html

In [ ]:
# check the dataframe
weather_df.show()

In [ ]:
# find the missing values
missing_values = (weather_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in weather_df.columns]))
missing_values.show()

In [ ]:
# select the row with missing values
input_cols = ["air_temperature","cloud_coverage","dew_temperature","sea_level_pressure","wind_direction","wind_speed"]
# define the imputer
imputer = Imputer(inputCols=input_cols, outputCols=input_cols).setStrategy("mean")
# apply to weather dataframe
weather_imputed = imputer.fit(weather_df).transform(weather_df)
# check the result
missing_values = (weather_imputed.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in weather_df.columns]))
missing_values.show()

We know that different seasons may affect energy consumption—for instance, a heater in winter and a cooler in summer. Extracting peak seasons (summer and winter) or off-peak seasons (Spring and Autumn) might be more useful than directly using the month as numerical values.   
1.2.3 The dataset has 16 sites in total, whose locations may span across different countries. Add a column (peak/off-peak) to the weather data frame based on the average air temperature. The top 3 hottest months and the 3 coldest months are considered “peak”, and the rest of the year is considered “off-peak”. 

In [ ]:
# find month from timestamp
weather_month = weather_imputed.withColumn("month", F.month("timestamp"))

# calculate the avg temp
monthly_avg_temp = (weather_month.groupBy("month").agg(F.avg("air_temperature").alias("avg_temp"))
                    .orderBy("avg_temp", ascending=False))
# find the hottest and coldest
hottest_months = [row["month"] for row in monthly_avg_temp.take(3)] # first 3 with descending are the hottest
coldest_months = [row["month"] for row in monthly_avg_temp.orderBy("avg_temp").take(3)] # first 3 with ascending are the coldest

# add (peak/off-peak) column
weather_labeled = weather_month.withColumn("peak/off-peak",F.when(F.col("month").isin(hottest_months + coldest_months), "peak")
                                           .otherwise("off-peak"))
# check the result
weather_labeled.show(10)

Create a data frame with all relevant columns at this stage, we refer to this data frame as feature_df.

In [ ]:
# check building df to see how to join
buildings_df.show()

In [ ]:
# dataframe of meters and buildings are joined by building_id 
feature_df = meters_labeled.join(buildings_df, on="building_id", how="left")

In [ ]:
# create aggregate 6-hour intervals for waether to join with the same time period
weather_label = (
    weather_labeled  
    .withColumn("year", F.year("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withColumn("day", F.dayofmonth("timestamp"))
    .withColumn("hour", F.hour("timestamp"))
    .withColumn(
        "time_interval",
        F.when((F.col("hour") >= 0) & (F.col("hour") < 6), "0:00-5:59")
         .when((F.col("hour") >= 6) & (F.col("hour") < 12), "6:00-11:59")
         .when((F.col("hour") >= 12) & (F.col("hour") < 18), "12:00-17:59")
         .otherwise("18:00-23:59")
    )
    .groupBy("site_id", "year", "month", "day", "time_interval", "peak/off-peak")
    .agg(
        F.avg("air_temperature").alias("air_temperature"),
        F.avg("cloud_coverage").alias("cloud_coverage"),
        F.avg("dew_temperature").alias("dew_temperature"),
        F.avg("sea_level_pressure").alias("sea_level_pressure"),
        F.avg("wind_direction").alias("wind_direction"),
        F.avg("wind_speed").alias("wind_speed")
    )
)

In [ ]:
weather_label.show(5)

In [ ]:
# join weather dataframe
feature_df = feature_df.join(weather_label,on=["site_id","year","month","day","time_interval"],how="left")

In [ ]:
feature_df.show()

### 1.3 Exploring the Data <a class="anchor" name="1.3"></a>
You can use either the CDA or the EDA method mentioned in Lab 5.  
Some ideas for CDA:  
a)	Older building may not be as efficient as new ones, therefore need more energy for cooling/heating. It’s not necessarily true though, if the buildings are built with higher standard or renovated later.  
b)	A multifloored or larger building obviously consumes more energy.  

1.	With the feature_df, write code to show the basic statistics:  
a) For each numeric column, show count, mean, stddev, min, max, 25 percentile, 50 percentile, 75 percentile;  
b) For each non-numeric column, display the top-5 values and the corresponding counts;  
c) For each boolean column, display the value and count. (note: pandas describe is allowed for this task.) (5%)

In [ ]:
# check the missing value
missing_values = (feature_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in feature_df.columns]))
missing_values.show()

In [ ]:
# fill peak/off-peak by month
feature_df = feature_df.withColumn("peak/off-peak",F.when(F.col("month").isin(hottest_months + coldest_months), "peak")
                                           .otherwise("off-peak"))
# fill other weather data with mean
weather_cols = ["air_temperature","cloud_coverage","dew_temperature","sea_level_pressure","wind_direction","wind_speed"]
imputer = Imputer(inputCols=weather_cols, outputCols=weather_cols).setStrategy("mean")
feature_df = imputer.fit(feature_df).transform(feature_df)
# check missing value again
missing_values = (feature_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in feature_df.columns]))
missing_values.show()

In [ ]:
feature_df.dtypes

### numeric column

In [ ]:
# transfer to pandas
pd_feature_df = feature_df.toPandas()

In [ ]:
# show the stats
stats = pd_feature_df.describe(percentiles=[.25, .5, .75]).T
print(stats)

In [ ]:
# remove the outliers in energy_consumption by std
print("Row count before remove outliers:", feature_df.count())
# get mean and std value
stats = feature_df.select( F.mean("energy_consumption"), F.stddev("energy_consumption")).first()
mean_val, std_val = stats[0], stats[1]
# remove the outliers
feature_df = feature_df.filter(
    (F.col("energy_consumption") <= mean_val + 3*std_val) &
    (F.col("energy_consumption") >= mean_val - 3*std_val)
)
print("Row count after remove outliers:", feature_df.count())

### non-numeric column

In [ ]:
categorical_cols = ["primary_use", "time_interval","meter_type"]
# print top 5 values and their counts
for col in categorical_cols:
    print(f"Top 5 values for {col}:")
    feature_df.groupBy(col).count().orderBy(F.desc("count")).show(5)
    
# peak and off-peak column
print(f"values for peak/off-peak:",)
feature_df.groupBy("peak/off-peak").count().show()

2.	Explore the dataframe and write code to present two plots of multivariate analysis, describe your plots and discuss the findings from the plots. (5% each).  
○	150 words max for each plot’s description and discussion.  
○	Note: In the building metadata table, there are some latent columns (data that may or may not be helpful, their meanings is unknown due to privacy and data security concerns).  
○	Feel free to use any plotting libraries: matplotlib, seabon, plotly, etc. You can refer to https://samplecode.link  


In [ ]:
# transfer to pandas
pd_feature_df = feature_df.toPandas()

### Discussion about Plot 1
This scatterplot shows the relationship between building decade and energy consumption, with colors hue indicating primary use and size representing square feet.

We can observe that the newer buildings, especially those constructed after 2000, show higher energy consumption, often driven by their usage and size, such as large office and education facilities. On the other hand, older buildings generally have lower consumption, partly due to smaller scale and less energy-intensive functions. Over the decades, residential and community buildings remain low in energy usage, while office and service buildings dominate the high-consumption range. 

Overall, we can see that not only built decades, building function and size also show a stronger influence on energy consumption. However, we cannot fully trust the results, as some buildings may have higher standards or have been renovated later.

In [ ]:
# group by year built in decades
pd_feature_df["decade"] = (pd_feature_df["year_built"] // 10) * 10 

In [ ]:
# visualization 1
# create scatterplot for decades to see the relationship between energy consumption with size and use
plt.figure(figsize=(14,6))
sns.scatterplot(
    data=pd_feature_df,
    x="decade",
    y="energy_consumption",
    hue="primary_use",
    size="square_feet",
    alpha=0.6
)
plt.title("Energy Consumption vs Building Decade with size and use")
plt.xlabel("Building Decade")
plt.ylabel("Energy Consumption")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.)
plt.tight_layout()
plt.show()

### Discussion about Plot 2 
This chart compares the average energy consumption (bars) and average air temperature (line) across 16 sites.  

We can observe the significant variation in energy consumption, with sites 7 and 13 recording the highest levels, while sites 5, 8, and 10 show much lower consumption. Furthermore, high energy use does not always correlate with temperature. For example, while the most two consumed sites are all colder sites overall, some colder sites, such as site 11, exhibit relatively low consumption, and also warmer sites do not consume more energy due to the cooling demand. 

Overall, the findings from this visualization suggest that while climate has some influence, it is not the primary factor.  Instead, site-specific characteristics such as building density, primary use, and operational requirements might play a more significant role in determining total energy consumption.

In [ ]:
# get average temperature for each site
avg_temp_site = (pd_feature_df.groupby("site_id").agg(avg_energy=("energy_consumption", "mean"),
                                                 avg_temp=("air_temperature", "mean")).reset_index())

In [ ]:
# visualization 2

# create chart see the relationships between energy consumption and each site and its avg temp
fig, ax1 = plt.subplots(figsize=(12,6))

# energy comsumption for each site
sns.barplot(
    data=avg_temp_site,
    x="site_id",
    y="avg_energy",
    color="steelblue",
    ax=ax1
)
ax1.set_ylabel("Average Energy Consumption")
ax1.set_xlabel("Site ID")
ax1.set_title("Average Energy Consumption and Air Temperature per Site")

# temperature for each site
ax2 = ax1.twinx()
sns.lineplot(
    data=avg_temp_site,
    x="site_id",
    y="avg_temp",
    marker="o",
    color="orange",
    ax=ax2
)
ax2.set_ylabel("Average Air Temperature")

plt.show()

## Part 2. Feature extraction and ML training <a class="anchor" name="part-2"></a>
In this section, you must use PySpark DataFrame functions and ML packages for data preparation, model building, and evaluation. Other ML packages, such as scikit-learn, should not be used to process the data; however, it’s fine to use them to display the result or evaluate your model.  
### 2.1 Discuss the feature selection and prepare the feature columns

2.1.1 Based on the data exploration from 1.2 and considering the use case, discuss the importance of those features (For example, which features may be useless and should be removed, which feature has a significant impact on the label column, which should be transformed), which features you are planning to use? Discuss the reasons for selecting them and how you plan to create/transform them.  
○	300 words max for the discussion  
○	Please only use the provided data for model building  
○	You can create/add additional features based on the dataset  
○	Hint - Use the insights from the data exploration/domain knowledge/statistical models to consider whether to create more feature columns, whether to remove some columns  

###  discussion
numerical features: all variables will be retained. Square_feet feature shows a highly skewed distribution, and therefore a log transformation will be performed to reduce the effect of extreme values. Other numerical variables will be preserved in their original form.

non-numerical data: primary_use and meter_type will be treated as categorical variables and transformed using one-hot encoding. Both time_interval and wind_direction are cyclical in nature and will be encoded using sine and cosine transformations to preserve their periodic properties.

date variables: year column will be removed as it only contains 2022. Although peak/off-peak labels already capture seasonal effects, the month variable will also be preserved through cyclical encoding (sine and cosine) to retain finer seasonal variations. The day column will be dropped due to limited predictive power and the high dimensionality that one-hot encoding would generate. The year_built column will be aggregated into decades units to capture chronological effects while avoiding noise.

latent features: latent_y, latent_s, and latent_r, their meaning is unclear. We will include them in the model for testing first, and then decide whether to remove them.

identifiers: such as row_id and building_id will be excluded since they provide no explanatory power and may lead to overfitting.

2.1.2 Write code to create/transform the columns based on your discussion above.

In [ ]:
# === Model spine now delegates to the shared energy_pipeline package ===
# Build the labelled, feature-engineered training frame from the SINGLE shared
# implementation. datasets.build_training_frame aggregates meters/weather to a
# real 6-hour interval_start timestamp and calls features.engineer_features, so
# time-of-day features use floor(hour / 6) -- identical to the streaming job.
# This removes the earlier string-vs-int time_interval skew.
model_frame = datasets.build_training_frame(spark, config)
model_frame.printSchema()

In [ ]:
# Imputation, the primary_use AND meter_type StringIndexer/OneHotEncoder and the
# VectorAssembler are all stages of the SHARED ML pipeline
# (features.build_estimator_pipeline). Folding them into the persisted model makes
# it self-contained: the streaming job loads it and only needs the raw inputs.
# meter_type is kept as a predictor and reconstructed in the stream via a
# cross-join (notebook 03 / design.md Decision 4).

In [ ]:
# (Cyclical time / wind / month encodings live inside the shared module:
# features.engineer_features for the deterministic parts, plus a post-imputation
# SQLTransformer stage for wind direction. No per-notebook copy to drift.)

In [ ]:
# (Building features -- log_square_feet, decade -- and the month encodings are
# also produced by features.engineer_features.)

In [ ]:
# (Identifier columns are simply ignored by the model: only
# features.ASSEMBLER_INPUTS feed the VectorAssembler, so no manual drops needed.)

In [ ]:
print("Training-frame columns:", model_frame.columns)

### 2.2 Preparing Spark ML Transformers/Estimators for features, labels, and models  <a class="anchor" name="2.2"></a>

**2.2.1 Write code to create Transformers/Estimators for transforming/assembling the columns you selected above in 2.1 and create ML model Estimators for Random Forest (RF) and Gradient-boosted tree (GBT) model.
Please DO NOT fit/transform the data yet.**

In [ ]:
# The feature space (VectorAssembler inputs) is defined once in
# energy_pipeline.features.ASSEMBLER_INPUTS and includes BOTH primary_use_ohe and
# meter_type_ohe. The assembler is a pipeline stage, so we do not build it here.
labelCol = config.model.label_col

**2.2.2. Write code to include the above Transformers/Estimators into two pipelines.
Please DO NOT fit/transform the data yet.**

In [ ]:
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor

# Estimators wrapped in the SHARED preprocessing pipeline
# (Imputer -> wind SQL -> indexers -> one-hot encoders -> assembler -> estimator).
rf = RandomForestRegressor(featuresCol="features", labelCol=labelCol, maxDepth=3, numTrees=10)
gbt = GBTRegressor(featuresCol="features", labelCol=labelCol, maxDepth=3, maxIter=10)
rf_pipeline = features.build_estimator_pipeline(rf)
gbt_pipeline = features.build_estimator_pipeline(gbt)

### 2.3 Preparing the training data and testing data  
Write code to split the data for training and testing, using 2025 as the random seed. You can decide the train/test split ratio based on the resources available on your laptop.  
Note: Due to the large dataset size, you can use random sampling (say 20% of the dataset). 

In [ ]:
# Sample/split the shared training frame. Outlier bounds are fit on the TRAINING
# split only and applied to training data, so no test-set statistics leak into
# the fit (datasets.compute_outlier_bounds / filter_outliers).
frame = model_frame
if config.model.sample_fraction < 1.0:
    frame = frame.sample(
        withReplacement=False, fraction=config.model.sample_fraction, seed=config.model.seed
    )

train_df, test_df = frame.randomSplit(
    [config.model.train_ratio, 1 - config.model.train_ratio], seed=config.model.seed
)
lo, hi = datasets.compute_outlier_bounds(train_df, labelCol, config.model.outlier_sigma)
train_df = datasets.filter_outliers(train_df, labelCol, lo, hi)
train_df.cache()

### 2.4 Training and evaluating models  
2.4.1 Write code to use the corresponding ML Pipelines to train the models on the training data from 2.3. And then use the trained models to predict the testing data from 2.3

In [ ]:
# train rf pipeline
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

# train GBT pipeline
gbt_model = gbt_pipeline.fit(train_df)
gbt_predictions = gbt_model.transform(test_df)

2.4.2 For both models (RF and GBT): with the test data, decide on which metrics to use for model evaluation and discuss which one is the better model (no word limit; please keep it concise). You may also use a plot for visualisation (not mandatory).

In [ ]:
# Evaluate both models with the SHARED metrics (RMSLE clamps negatives before
# log1p, matching the project's metrics module).
import json

print("RandomForest:", json.dumps(metrics.evaluate(rf_predictions, labelCol)))
print("GBT         :", json.dumps(metrics.evaluate(gbt_predictions, labelCol)))

After evaluation above, all metrics in the GBT model outperformed those in the Random Forest model. Therefore, I decided to save the GBT model.

2.4.3 3.	Save the better model (you’ll need it for the streaming pipeline).
(Note: You may need to go through a few training loops or use more data to create a better-performing model.)

In [ ]:
# Save the better model (GBT) to the shared artifact path the streaming job reads.
gbt_model.write().overwrite().save(config.paths.model)

### Part 3. Hyperparameter Tuning and Model Optimisation <a class="anchor" name="part-3"></a>  
Apply the techniques you have learnt from the labs, for example, CrossValidator, TrainValidationSplit, ParamGridBuilder, etc., to perform further hyperparameter tuning and model optimisation.  
The assessment is based on the quality of your work/process, not the quality of your model. Please include your thoughts/ideas/discussions.

#### Discussion about Hyperparameter Tuning Process:

From the previous stage, I choosed Gradient-Boosted Trees as the base model and applied CrossValidator with 3-fold cross-validation to search for the best hyperparameters. The tuning process aimed to reduce the RMSLE on the validation set.

#### Parameters tuned and tested ranges:
Overall, the parameter ranges were kept relatively small in order to control model training time and avoid memory overload during cross-validation.

`maxDepth`: [2, 4, 6]  
Rational: Tune the parameter of maxDepth to find the best parameter that can catch enough interactions and won't overfit.

`maxIter`: [10, 20, 50]  
Rational: Tune the parameter of maxIter to get a balance between accuracy and computational cost.

`stepSize`: [0.05, 0.1, 0.2] 
Rational: Tune the parameter of stepSize to explore the trade-off between stability and efficiency. 

#### Evaluation metric:
Since `RegressionEvaluator` in PySpark does not support RMSLE directly, we first used RMSE as the evaluation metric during cross-validation to identify the best hyperparameters. After selecting the best model, we then applied a custom RMSLE function on the test dataset to evaluate its performance.

#### Results and discussion:
The best model was obtained with `maxDepth=4`, `maxIter=50`, and `stepSize=0.05`.  

Then, we apply this model on the test set, we can see that compared to the baseline GBT model, the RMSLE improved from **2.57 to 2.42**, indicating that moderately deep trees with more iterations and a smaller learning rate can capture additional variance while avoiding overfitting.  

In summary, hyperparameter tuning improved the model’s predictive accuracy and showed the importance of get a balance between model complexity, convergence stability, and generalisation when we are creating a model.


In [ ]:
# create ParamGrid for Cross Validation
# NOTE: trimmed grid (4 combos x 3 folds = 12 GBT fits) so it completes on a
# single-machine local[*] run. Original 27-combo grid (81 fits) was intractable here.
gbtParamGrid = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.maxIter, [10, 20])
    .addGrid(gbt.stepSize, [0.1])
    .build())

In [ ]:
# define an evaluator for evaluating the model 
gbtEvaluator = RegressionEvaluator(
    labelCol="energy_consumption",
    predictionCol="prediction",
    metricName="rmse"
)

In [ ]:
# 3-fold Cross Validation
gbtCV = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=gbtParamGrid,
    evaluator=gbtEvaluator,  
    numFolds=3,
    seed=2025 # # fix the outcome
)

In [ ]:
# Run cross validations
gbtCVModel = gbtCV.fit(train_df)

In [ ]:
# get the best model and check the result
best_model = gbtCVModel.bestModel
print('Best Param for GBT: ', best_model.stages[-1]._java_obj.paramMap())

In [ ]:
# Evaluate the tuned model on the held-out test set with the shared metrics.
best_scores = metrics.evaluate(gbtCVModel.transform(test_df), labelCol)
print("Tuned GBT metrics:", best_scores)

In [ ]:
best_model.write().overwrite().save(config.paths.model)